In [1]:
# ================================================================
# SENTIMENT MODEL CELL 1 — Imports
# ================================================================
from textblob import TextBlob
import re
import pickle

print('Imports done!')

Imports done!


In [2]:
# ================================================================
# SENTIMENT MODEL CELL 2 — Load Model
# ================================================================
vectorizer = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detector\news\AI\Text Models\Text Model\tfidf_vectorizer.pkl', 'rb'))
best_model = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detector\news\AI\Text Models\Text Model\best_fake_news_model.pkl', 'rb'))

print('Model loaded!')

Model loaded!


In [3]:
# ================================================================
# SENTIMENT MODEL CELL 3 — Sensationalism Detector
# Checks for fake news language patterns
# ================================================================

# Words commonly found in fake news
SENSATIONAL_WORDS = [
    'breaking', 'shocking', 'bombshell', 'explosive', 'urgent',
    'alert', 'exposed', 'leaked', 'confirmed', 'proof', 'secret',
    'censored', 'banned', 'deleted', 'share', 'wake up', 'sheeple',
    'deep state', 'cover up', 'cover-up', 'mainstream media',
    'they dont want', 'they don\'t want', 'before they delete',
    'share before', 'wake up', 'open your eyes', 'truth',
    'massive', 'huge', 'incredible', 'unbelievable', 'disgusting'
]

def check_sensationalism(text):
    text_lower = text.lower()
    
    # Count sensational words found
    found_words = [w for w in SENSATIONAL_WORDS if w in text_lower]
    
    # Count exclamation marks
    exclamation_count = text.count('!')
    
    # Count ALL CAPS words
    words = text.split()
    caps_words = [w for w in words if w.isupper() and len(w) > 2]
    
    # Calculate sensationalism score (0-100)
    score = 0
    score += min(len(found_words) * 10, 40)   # max 40 from sensational words
    score += min(exclamation_count * 5, 30)    # max 30 from exclamation marks
    score += min(len(caps_words) * 5, 30)      # max 30 from caps words
    score = min(score, 100)                    # cap at 100
    
    return {
        'score'            : score,
        'sensational_words': found_words,
        'exclamation_marks': exclamation_count,
        'caps_words'       : caps_words[:5],   # show first 5 only
    }

print('check_sensationalism() is ready!')

check_sensationalism() is ready!


In [4]:
# ================================================================
# SENTIMENT MODEL CELL 4 — Full Sentiment Analysis
# ================================================================
def analyze_sentiment(text):
    # TextBlob sentiment
    blob = TextBlob(text)
    
    # Polarity: -1 (very negative) to +1 (very positive)
    polarity = blob.sentiment.polarity
    
    # Subjectivity: 0 (objective/factual) to 1 (subjective/opinion)
    subjectivity = blob.sentiment.subjectivity
    
    # Convert polarity to label
    if polarity > 0.2:
        polarity_label = 'Positive 😊'
    elif polarity < -0.2:
        polarity_label = 'Negative 😡'
    else:
        polarity_label = 'Neutral 😐'
    
    # Convert subjectivity to label
    if subjectivity > 0.6:
        subjectivity_label = 'Very Subjective (Opinion based) ⚠️'
    elif subjectivity > 0.4:
        subjectivity_label = 'Somewhat Subjective'
    else:
        subjectivity_label = 'Objective (Fact based) ✅'
    
    return {
        'polarity'          : round(polarity, 2),
        'polarity_label'    : polarity_label,
        'subjectivity'      : round(subjectivity, 2),
        'subjectivity_label': subjectivity_label,
    }

print('analyze_sentiment() is ready!')

analyze_sentiment() is ready!


In [5]:
# ================================================================
# SENTIMENT MODEL CELL 5 — Predict from text
# ================================================================
def predict_news(text, confidence_threshold=0.40):
    text_clean = re.sub(r'^[A-Z\s,\.]+\(Reuters\)\s*[-\u2013]\s*', '', str(text))
    text_tfidf = vectorizer.transform([text_clean])
    if text_tfidf.nnz == 0:
        return 'Uncertain (No known words) ❓', 50
    prob = best_model.predict_proba(text_tfidf)[0]
    pred = best_model.predict(text_tfidf)[0]
    confidence = prob[pred]
    if confidence < confidence_threshold:
        return f'Uncertain (Confidence: {confidence:.2f}) ❓', 50
    label = 'Real 🟢' if pred == 1 else 'Fake 🔴'
    score = int(confidence * 100) if pred == 1 else int((1 - confidence) * 100)
    return label, score

print('predict_news() is ready!')

predict_news() is ready!


In [6]:
# ================================================================
# SENTIMENT MODEL CELL 6 — Full Analysis Function
# ================================================================
def analyze_sentiment_full(text):
    print('=' * 60)
    print('         SENTIMENT & EMOTION ANALYSIS')
    print('=' * 60)

    # Step 1 — Sentiment
    sentiment = analyze_sentiment(text)
    print(f'\n📊 SENTIMENT ANALYSIS:')
    print(f'   Polarity:     {sentiment["polarity_label"]} ({sentiment["polarity"]})')
    print(f'   Subjectivity: {sentiment["subjectivity_label"]} ({sentiment["subjectivity"]})')

    # Step 2 — Sensationalism
    sensational = check_sensationalism(text)
    print(f'\n🚨 SENSATIONALISM CHECK:')
    print(f'   Score:        {sensational["score"]}/100')
    print(f'   Exclamations: {sensational["exclamation_marks"]}')
    print(f'   CAPS words:   {sensational["caps_words"]}')
    print(f'   Trigger words found: {sensational["sensational_words"][:5]}')

    # Step 3 — Text model prediction
    prediction, confidence = predict_news(text)
    print(f'\n🤖 TEXT MODEL:')
    print(f'   Prediction:   {prediction}')

    # Step 4 — Combined sentiment score
    # High subjectivity + high sensationalism = likely fake
    sentiment_fake_score = int(
        sentiment['subjectivity'] * 40 +      # 40% weight on subjectivity
        (sensational['score'] / 100) * 60      # 60% weight on sensationalism
    )

    if sentiment_fake_score >= 70:
        sentiment_verdict = 'Likely Fake 🔴'
    elif sentiment_fake_score >= 40:
        sentiment_verdict = 'Uncertain ⚠️'
    else:
        sentiment_verdict = 'Likely Real 🟢'

    print(f'\n{"=" * 60}')
    print(f'  SENTIMENT VERDICT: {sentiment_verdict}')
    print(f'  Sentiment Score:   {sentiment_fake_score}/100 fake indicators')
    print(f'  Text Model:        {prediction}')
    print(f'{"=" * 60}')

    return {
        'sentiment'       : sentiment,
        'sensationalism'  : sensational,
        'text_prediction' : prediction,
        'sentiment_score' : sentiment_fake_score,
        'verdict'         : sentiment_verdict
    }

print('analyze_sentiment_full() is ready!')

analyze_sentiment_full() is ready!


In [7]:
# ================================================================
# SENTIMENT MODEL CELL 7 — Test it
# ================================================================
# Test with fake news
fake_text = "BREAKING!! Hillary Clinton ARRESTED by FBI!! Sources confirm Obama involved in MASSIVE COVER-UP!! Share before they DELETE THIS!!"
print('--- FAKE NEWS TEST ---')
fake_result = analyze_sentiment_full(fake_text)

print('\n\n')

# Test with real news
real_text = "The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, citing continued progress toward its inflation target."
print('--- REAL NEWS TEST ---')
real_result = analyze_sentiment_full(real_text)

--- FAKE NEWS TEST ---
         SENTIMENT & EMOTION ANALYSIS

📊 SENTIMENT ANALYSIS:
   Polarity:     Neutral 😐 (0.0)
   Subjectivity: Very Subjective (Opinion based) ⚠️ (1.0)

🚨 SENSATIONALISM CHECK:
   Score:        100/100
   Exclamations: 8
   CAPS words:   ['BREAKING!!', 'ARRESTED', 'FBI!!', 'MASSIVE', 'COVER-UP!!']
   Trigger words found: ['breaking', 'share', 'cover-up', 'before they delete', 'share before']

🤖 TEXT MODEL:
   Prediction:   Fake 🔴

  SENTIMENT VERDICT: Likely Fake 🔴
  Sentiment Score:   100/100 fake indicators
  Text Model:        Fake 🔴



--- REAL NEWS TEST ---
         SENTIMENT & EMOTION ANALYSIS

📊 SENTIMENT ANALYSIS:
   Polarity:     Neutral 😐 (0.0)
   Subjectivity: Objective (Fact based) ✅ (0.0)

🚨 SENSATIONALISM CHECK:
   Score:        0/100
   Exclamations: 0
   CAPS words:   []
   Trigger words found: []

🤖 TEXT MODEL:
   Prediction:   Real 🟢

  SENTIMENT VERDICT: Likely Real 🟢
  Sentiment Score:   0/100 fake indicators
  Text Model:        Real 🟢
